In [1]:
import os
import torch
import time
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from skimage.color import rgb2lab, lab2rgb
from PIL import Image
from fastai.vision.learner import create_body
import torchvision.models as models
from fastai.vision.models.unet import DynamicUnet
import matplotlib.pyplot as plt
import yaml
import random
import pandas as pd
from copy import deepcopy
import scienceplots
from glob import glob
from datetime import datetime

# !pip install latex
# sudo apt-get install texlive-latex-extra texlive-fonts-recommended dvipng cm-super
plt.style.use('science')

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'[INFO] Device: {device} - Torch version: {torch.__version__}')

[INFO] Device: cuda - Torch version: 2.3.0+cu118


In [3]:
EXPERIMENT_NAME = 'simple-unet-shufflenet_v2_x1_5'
SEED = 1337
IMAGE_SIZE = 384
BATCH_SIZE = 8
PATIENCE = 20
N_WORKERS = max(0, os.cpu_count() - 4)
DATASET_PATH = './'
BACKBONE = 'shufflenet_v2_x1_5'
SELF_ATTENTION = True
ACTIVATION_FUNCTION = 'Mish'
CRITERION = torch.nn.L1Loss()
OPTIM = torch.optim.Adam
LR = 1e-4
EPOCHS = 200

In [4]:
experiment_path = F'./experiments/{EXPERIMENT_NAME}-{datetime.now().strftime("%Y-%m-%d-%H:%M:%S")}'
os.makedirs(experiment_path, exist_ok=True)

In [5]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
class SEMColorizationDataset(Dataset):
    def __init__(self, file_paths, image_size, train=True):
    
        self.file_paths = file_paths

        if train:
            self.transforms = transforms.Compose([
                transforms.Resize((image_size, image_size), Image.BICUBIC),
                transforms.RandomVerticalFlip(p=0.2),
                transforms.RandomHorizontalFlip(p=0.2),
                transforms.RandomPerspective(distortion_scale=0.1),
                transforms.RandomRotation(degrees=10),
            ])
        else:
            self.transforms = transforms.Resize((image_size, image_size),  Image.BICUBIC)

    
    def rgb_to_lab(self, img: np.ndarray):

        img_lab = rgb2lab(np.array(img))
    
        # Reshape to (image_size, image_size, channels)
        img_lab = torch.from_numpy(img_lab).permute(2, 0, 1).float()
        img_lab = torch.unsqueeze(img_lab, 1)

        L = img_lab[0]
        ab = img_lab[1:]

        # Normalization: -1.0 <= x <= 1.0
        L = (L / 50.0) - 1.0
        ab = (ab + 128.0) / 255.0
        
        return {'L': L, 'ab': ab}
    

    def lab_to_rgb(self, L, ab):
        """
        Takes a batch of images in the Lab color space and converts them to RGB.
        """
        
        # Denormalize
        L = (L + 1.0) * 50.0
        ab = ab * 255.0 - 128.0
        Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
        rgb_imgs = []

        for img in Lab:
            img_rgb = lab2rgb(img)
            rgb_imgs.append(img_rgb)

        return np.stack(rgb_imgs, axis=0)
    

    def __getitem__(self, idx):
        
        img = Image.open(self.file_paths[idx]).convert('RGB')
        img = self.transforms(img)
        
        return self.rgb_to_lab(img)
    

    def __len__(self):
        return len(self.file_paths)

In [7]:
def get_dataloaders(batch_size=32, 
                    n_workers=4, 
                    pin_memory=True, 
                    file_paths=None,
                    image_size=256,
                    train=True):
    
    dataset = SEMColorizationDataset(file_paths=file_paths, 
                                     image_size=image_size,
                                     train=train)

    dataloader = DataLoader(dataset, 
                            batch_size=batch_size, 
                            num_workers=n_workers,
                            pin_memory=pin_memory)
    
    return dataloader

In [8]:
train_paths = glob(f'{DATASET_PATH}/train/*.png')  \
            + glob(f'{DATASET_PATH}/train/*.jpeg') \
            + glob(f'{DATASET_PATH}/train/*.jpg') 

val_paths = glob(f'{DATASET_PATH}/val/*.png') \
          + glob(f'{DATASET_PATH}/val/*.jpeg') \
          + glob(f'{DATASET_PATH}/val/*.jpg') 

train_dl = get_dataloaders(batch_size=BATCH_SIZE, 
                           n_workers=N_WORKERS, 
                           file_paths=train_paths, 
                           image_size=IMAGE_SIZE)

val_dl = get_dataloaders(batch_size=BATCH_SIZE, 
                         n_workers=N_WORKERS, 
                         file_paths=val_paths, 
                         image_size=IMAGE_SIZE, 
                         train=False)

In [9]:
def build_model(backbone, n_input=1, n_output=2, size=256):

    if backbone == 'resnet18':
        backbone = models.resnet18(weights='DEFAULT')
    
    elif backbone == 'resnet34':
        backbone = models.resnet34(weights='DEFAULT')
    
    elif backbone == 'convnext_tiny':
        backbone = models.convnext_tiny(weights='DEFAULT')

    elif backbone == 'efficientnet_v2_s':
        backbone = models.efficientnet_v2_s(weights='DEFAULT')

    elif backbone == 'shufflenet_v2_x0_5':
        backbone = models.shufflenet_v2_x2_0(weights='DEFAULT')

    elif backbone == 'shufflenet_v2_x1_0':
        backbone = models.shufflenet_v2_x2_0(weights='DEFAULT')

    elif backbone == 'shufflenet_v2_x1_5':
        backbone = models.shufflenet_v2_x2_0(weights='DEFAULT')

    elif backbone == 'shufflenet_v2_x2_0':
        backbone = models.shufflenet_v2_x2_0(weights='DEFAULT')
  
    body = create_body(backbone, n_in=n_input, pretrained=True, cut=-2) 

    if ACTIVATION_FUNCTION:
        model = DynamicUnet(body, n_output, (size, size), self_attention=SELF_ATTENTION, act_cls=torch.nn.Mish).to(device)
    else:
        model = DynamicUnet(body, n_output, (size, size), self_attention=SELF_ATTENTION).to(device)

    return model


In [10]:
def train_and_val(loss_values_train, 
                  loss_values_val, 
                  model, 
                  scaler,
                  train_dl, 
                  val_dl,
                  optim, 
                  criterion,  
                  epochs):

    torch.cuda.empty_cache()

    best_model = deepcopy(model.state_dict())

    min_val_loss = np.inf
    patience_counter = 0
    best_epoch = 0

    loss_train = np.inf
    loss_val = np.inf

    for i in range(epochs):

        loss_batch = 0.0
        total_samples = 0
        
        ### Train
        for data in tqdm(train_dl, desc = f"Epoch {i+1}/{epochs} | Loss Train: {loss_train:.6f}"):
            
            L, ab = data['L'].to(device), data['ab'].to(device)

            model.train()
            
            preds = model(L).unsqueeze(2)
            
            optim.zero_grad()

            # Casts operations to mixed precision
            with torch.amp.autocast(device_type=device, dtype=torch.float16):
                loss = criterion(preds, ab)

            # Scales the loss, and calls backward()
            # to create scaled gradients
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()

            loss_batch += loss.item() * L.size(0)  
            total_samples += L.size(0) 
        
        loss_train = loss_batch / total_samples
        loss_values_train.append(loss_train)

        loss_batch = 0.0
        total_samples = 0

        ### Validation
        for data in tqdm(val_dl, desc = f"Epoch {i+1}/{epochs} | Loss Val: {loss_val:.3f}"):
            L, ab = data['L'].to(device), data['ab'].to(device)
            
            model.eval()

            with torch.inference_mode():
                preds = model(L).unsqueeze(2)
                
            with torch.amp.autocast(device_type=device, dtype=torch.float16):
                loss = criterion(preds, ab)

            loss_batch += loss.item() * L.size(0)  
            total_samples += L.size(0) 
        
        loss_val = loss_batch / total_samples
        loss_values_val.append(loss_val)

        # Early stopping
        if loss_val < min_val_loss:
            best_model = deepcopy(model.state_dict())
            print(f'[INFO] New best model found. Updating state dict...')
            min_val_loss = loss_val
            patience_counter = 0 
            best_epoch = i
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"[INFO] Early stopping triggered at epoch {best_epoch}, best validation loss: {min_val_loss:.4f}")
                break

    return best_model, best_epoch

In [11]:
def save_train_results(model, 
                       experiment_path, 
                       backbone, 
                       loss_values_train,
                       loss_values_val,
                       best_epoch=0,
                       patience=0):

    model_path = f"{experiment_path}/model.pth"
    training_results_path = f"{experiment_path}/results-{backbone}.csv"

    model = model.half() 
    
    torch.save({
        'epoch': best_epoch,
        'model_state_dict': model.state_dict(),
        'val_loss': loss_values_val[-1],
        'patience': patience
    }, model_path)
    
    print(f'[INFO] Saved model weights to {model_path}.')

    df = pd.DataFrame({
        'train_loss': loss_values_train,
        'val_loss': loss_values_val
    })

    df.to_csv(training_results_path, index=False)
    print(f'[INFO] Saved training results to {model_path}.')

    with plt.style.context('science'):
        plt.figure()
        plt.title('Losses from train and validation')
        plt.plot(df.index, df['train_loss'], label='Train')
        plt.plot(df.index, df['val_loss'], label='Validation')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig(f'{experiment_path}/loss_plot.png')

In [12]:
model = build_model(backbone=BACKBONE, n_input=1, n_output=2, size=IMAGE_SIZE)

pytorch_total_params = sum(p.numel() for p in model.parameters())
print(f'[INFO] Total number of parameters: {pytorch_total_params}')

### Automatic Mixed Precision
scaler = torch.cuda.amp.GradScaler()

optim = OPTIM(model.parameters(), lr=LR)
criterion = CRITERION

loss_values_train = []
loss_values_val = []

[INFO] Total number of parameters: 73623529


In [13]:
hyperparams = {
    'EXPERIMENT_NAME': EXPERIMENT_NAME,
    'SEED': SEED,
    'IMAGE_SIZE': IMAGE_SIZE,
    'BATCH_SIZE': BATCH_SIZE,
    'PATIENCE': PATIENCE,
    'N_WORKERS': N_WORKERS,
    'DATASET_PATH': DATASET_PATH,
    'BACKBONE': BACKBONE,
    'SELF_ATTENTION': SELF_ATTENTION,
    'ACTIVATION_FUNCTION': ACTIVATION_FUNCTION,
    'CRITERION': CRITERION.__class__.__name__,
    'OPTIM': optim.__class__.__name__,
    'LR': LR,
    'EPOCHS': EPOCHS
}

with open(f'{experiment_path}/config.yaml', 'w') as config_file:
    yaml.dump_all([hyperparams], config_file)

In [14]:
tic = time.time()
best_model, best_epoch = train_and_val(loss_values_train,
                                        loss_values_val,
                                        model, 
                                        scaler,
                                        train_dl, 
                                        val_dl,
                                        optim, 
                                        criterion, 
                                        EPOCHS)

model.load_state_dict(best_model)

print(f'Training took {round((time.time() - tic)/60, 2)} minutes.')

save_train_results(model,
                   experiment_path,
                   BACKBONE,
                   loss_values_train,
                   loss_values_val,
                   best_epoch,
                   PATIENCE
                    )

print(f'[INFO] Finished training.')

Epoch 1/200 | Loss Train: inf: 100%|██████████| 25/25 [00:25<00:00,  1.00s/it]
Epoch 1/200 | Loss Val: inf: 100%|██████████| 7/7 [00:03<00:00,  2.04it/s]


[INFO] New best model found. Updating state dict...


Epoch 2/200 | Loss Train: 0.337809: 100%|██████████| 25/25 [00:23<00:00,  1.07it/s]
Epoch 2/200 | Loss Val: 0.103: 100%|██████████| 7/7 [00:03<00:00,  2.16it/s]
Epoch 3/200 | Loss Train: 0.078505: 100%|██████████| 25/25 [00:23<00:00,  1.05it/s]
Epoch 3/200 | Loss Val: 1.521: 100%|██████████| 7/7 [00:03<00:00,  2.16it/s]


[INFO] New best model found. Updating state dict...


Epoch 4/200 | Loss Train: 0.067161:  64%|██████▍   | 16/25 [00:16<00:09,  1.04s/it]


KeyboardInterrupt: 